# 03 BKT baseline on Junyi (C1 Adaptive Curriculum Engine)

Fits Bayesian Knowledge Tracing per exercise with **pyBKT** on the training students only, then predicts the held-out test students.

1. **Fit** prior, learn, guess and slip per exercise (EM, seed 42), training students only.
2. **Check** for degenerate exercises (guess + slip >= 1), which are replaced by the median.
3. **Forward pass** with the same update formula as `curriculum-service/app/domain/mastery.py`, compared against pyBKT's own predictions. If they agree, the service implements BKT correctly.
4. **Evaluate** on the test students: AUC (primary), accuracy, RMSE, against the majority-class baseline.
5. **Export** per-exercise parameters for notebook 04 (GAT features) and median values for the service defaults.

Needs notebooks 01 and 02. CPU is enough; the fit takes roughly 10 to 30 minutes on the full sample (`QUICK = True` for a 5-minute test run).

In [ ]:
# Where outputs are stored: "drive", "runtime" or "local" (see notebook 01).
STORAGE = "drive"
# True: 3,000 training and 1,000 test students, for a fast check that everything runs.
QUICK = False

In [ ]:
import sys
!{sys.executable} -m pip install -q pyBKT pyarrow "scikit-learn<1.8"

from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if not ON_COLAB:
    STORAGE = "local"

if STORAGE == "drive":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/adaptlearn-c1")
    except Exception as exc:
        print(f"Drive could not be mounted ({type(exc).__name__}: {exc}). Using runtime storage.")
        STORAGE = "runtime"
if STORAGE == "runtime":
    BASE = Path("/content/adaptlearn-c1")
elif STORAGE == "local":
    BASE = Path("../data").resolve()

PROCESSED = BASE / "junyi_processed"
OUT_DIR = PROCESSED / "bkt"
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert (PROCESSED / "interactions.parquet").exists(), "Run notebook 01 first (or upload its outputs)."
print("Storage:", STORAGE, "->", BASE)

## 1. Load the answers and the student split from notebook 01

In [ ]:
import json
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

inter = pd.read_parquet(PROCESSED / "interactions.parquet",
                        columns=["user_id", "exercise", "exercise_idx", "step", "correct"])
splits = json.loads((PROCESSED / "splits.json").read_text())
exercise_index = json.loads((PROCESSED / "exercise_index.json").read_text())

train_ids = np.array(sorted(u for fold in splits["folds"].values() for u in fold))
test_ids = np.array(splits["test"])
if QUICK:
    train_ids = rng.choice(train_ids, 3_000, replace=False)
    test_ids = rng.choice(test_ids, 1_000, replace=False)

# Sorted by student, then time; order_id is a global running number pyBKT can sort on.
inter = inter.sort_values(["user_id", "step"], kind="mergesort").reset_index(drop=True)
inter["order_id"] = np.arange(len(inter))
train = inter[inter.user_id.isin(train_ids)].reset_index(drop=True)
test = inter[inter.user_id.isin(test_ids)].reset_index(drop=True)
assert not set(train.user_id) & set(test.user_id), "a test student is in training"
print(f"train: {train.user_id.nunique()} students, {len(train):,} answers")
print(f"test:  {test.user_id.nunique()} students, {len(test):,} answers")

## 2. Fit BKT per exercise (training students only)

In [ ]:
import time
from pyBKT.models import Model

# pyBKT is unmaintained and breaks on numpy >= 2.4 (Colab ships 2.0.x).
assert tuple(int(x) for x in np.__version__.split(".")[:2]) < (2, 4), (
    "Run: !pip install -q numpy==2.0.2   then Runtime > Restart session, and run again.")

def as_pybkt(df):
    # pyBKT's default column names; predict() does not accept a column mapping.
    return df.rename(columns={"exercise": "skill_name"})[["order_id", "user_id", "skill_name", "correct"]]

model = Model(seed=SEED, num_fits=1)
start = time.time()
model.fit(data=as_pybkt(train))
print(f"fitted {train.exercise.nunique()} exercises in {(time.time() - start) / 60:.1f} min")

raw = model.params().reset_index()
params = (raw[raw["param"].isin(["prior", "learns", "guesses", "slips"])]
          .pivot_table(index="skill", columns="param", values="value", aggfunc="first")
          .rename(columns={"prior": "p_init", "learns": "p_learn", "guesses": "p_guess", "slips": "p_slip"}))
params = params[["p_init", "p_learn", "p_guess", "p_slip"]]
params.describe().round(3)

## 3. Degenerate exercises

When guess + slip >= 1, a correct answer is more likely from a student who has **not** learned the exercise, so the model runs backwards. Those exercises get the median of the healthy ones. Exercises with no training answers get the median too.

In [ ]:
healthy = params[(params.p_guess + params.p_slip) < 1]
median = healthy.median()
degenerate = sorted(set(params.index) - set(healthy.index))
print(f"degenerate exercises: {len(degenerate)} of {len(params)}")

all_exercises = sorted(exercise_index, key=exercise_index.get)
final = params.copy()
final.loc[degenerate] = median.values
final = final.reindex(all_exercises)
unseen = final.p_init.isna().sum()
final = final.fillna(median)
final["status"] = "fitted"
final.loc[final.index.isin(degenerate), "status"] = "median (degenerate)"
final.loc[~final.index.isin(params.index), "status"] = "median (no training answers)"
final.index.name = "exercise"
print(f"exercises without training answers: {unseen}")
final.status.value_counts()

## 4. Forward pass with the service's formula

The same two steps as `app/domain/mastery.py`: Bayes' rule on the answer, then the learning step (every Junyi answer is practice). `mastery_before` is the probability the exercise is learned **before** the answer, so the prediction for answer *t* only uses answers before *t*.

In [ ]:
def bkt_forward(df: pd.DataFrame, p: pd.DataFrame) -> pd.DataFrame:
    ex = df["exercise_idx"].to_numpy()
    users = df["user_id"].to_numpy()
    y = df["correct"].to_numpy().astype(bool)
    by_idx = p.reindex(all_exercises)
    prior, learn = by_idx.p_init.to_numpy(), by_idx.p_learn.to_numpy()
    guess, slip = by_idx.p_guess.to_numpy(), by_idx.p_slip.to_numpy()

    before = np.empty(len(df))
    state = prior.copy()
    for i in range(len(df)):
        if i == 0 or users[i] != users[i - 1]:
            state = prior.copy()
        e = ex[i]
        m = state[e]
        before[i] = m
        if y[i]:
            post = m * (1 - slip[e]) / (m * (1 - slip[e]) + (1 - m) * guess[e])
        else:
            post = m * slip[e] / (m * slip[e] + (1 - m) * (1 - guess[e]))
        state[e] = post + (1 - post) * learn[e]
    p_correct = before * (1 - slip[ex]) + (1 - before) * guess[ex]
    return pd.DataFrame({"mastery_before": before, "p_correct": p_correct})

start = time.time()
pred = bkt_forward(test, final)
print(f"forward pass over {len(test):,} answers in {time.time() - start:.0f} s")

In [ ]:
# Cross-check against pyBKT's own predictions with its unmodified parameters,
# on exercises pyBKT has parameters for (skills are independent, so dropping others is safe).
known = test.exercise.isin(params.index).to_numpy()
pybkt_pred = model.predict(data=as_pybkt(test.loc[known]))
pybkt_pred = pybkt_pred.sort_values("order_id")["correct_predictions"].to_numpy()
own_raw = bkt_forward(test, params.reindex(all_exercises).fillna(median))["p_correct"].to_numpy()[known]
diff = np.abs(own_raw - pybkt_pred)
print(f"max |own - pyBKT| = {diff.max():.2e}, mean = {diff.mean():.2e}")
print("MATCH: the service formula equals pyBKT" if diff.max() < 1e-4
      else "MISMATCH: send this output to Claude before going on")

## 5. Results on the held-out test students

In [ ]:
from sklearn.metrics import accuracy_score, mean_squared_error, roc_auc_score

y_test = test["correct"].to_numpy()
majority = int(train["correct"].mean() >= 0.5)
results = pd.DataFrame([
    {"model": "majority class", "auc": 0.5,
     "accuracy": accuracy_score(y_test, np.full_like(y_test, majority)),
     "rmse": np.sqrt(mean_squared_error(y_test, np.full(len(y_test), train["correct"].mean())))},
    {"model": "BKT (pyBKT, per exercise)", "auc": roc_auc_score(y_test, pred.p_correct),
     "accuracy": accuracy_score(y_test, pred.p_correct >= 0.5),
     "rmse": np.sqrt(mean_squared_error(y_test, pred.p_correct))},
]).round(4)
results

## 6. Defaults for the service

The service starts every Java concept from one set of BKT values (it has no data of its own yet). Prior, learn and slip are the medians over the healthy Junyi exercises. Guess stays at 0.25 in the service, because every C1 item is a four-option multiple choice, unlike Junyi's open answers.

In [ ]:
service_defaults = {
    "p_init": round(float(median.p_init), 4),
    "p_learn": round(float(median.p_learn), 4),
    "p_slip": round(float(median.p_slip), 4),
    "p_guess_junyi_median": round(float(median.p_guess), 4),
    "p_guess_service": 0.25,
    "source": "median over healthy Junyi exercises, training students only",
}
print(json.dumps(service_defaults, indent=2))
print("\nPaste into app/domain/mastery.py:")
print(f'DEFAULT_PARAMS = BktParams(p_init={service_defaults["p_init"]}, p_learn={service_defaults["p_learn"]}, '
      f'p_guess=0.25, p_slip={service_defaults["p_slip"]}, version="bkt-junyi-v1")')

## 7. Save outputs and a manifest

In [ ]:
import platform
import sklearn
import pyBKT

final.to_csv(OUT_DIR / "params.csv")
(OUT_DIR / "service_defaults.json").write_text(json.dumps(service_defaults, indent=2))
(OUT_DIR / "metrics.json").write_text(results.to_json(orient="records", indent=2))
(OUT_DIR / "degenerate.json").write_text(json.dumps(degenerate))

CONFIG = {"SEED": SEED, "QUICK": QUICK, "num_fits": 1, "forgets": False,
          "train_students": int(len(train_ids)), "test_students": int(len(test_ids))}
VERSIONS = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
            "scikit-learn": sklearn.__version__, "pyBKT": getattr(pyBKT, "__version__", "unknown")}

In [ ]:
import hashlib, json, platform
from datetime import datetime, timezone

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG,
    "versions": VERSIONS,
    "sha256": {f.name: hashlib.sha256(f.read_bytes()).hexdigest()
               for f in sorted(OUT_DIR.glob("*")) if f.is_file() and f.name != "manifest.json"},
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

In [ ]:
if STORAGE == "runtime":
    import shutil
    print("Download before stopping the runtime:", shutil.make_archive("/content/bkt_outputs", "zip", OUT_DIR))
else:
    print("Outputs stored in", OUT_DIR)

## For the report

Report the results table (AUC is the primary metric; accuracy must be read against the 83.2% majority class), the number of degenerate exercises, the pyBKT cross-check and the service defaults. Notebook 04 reads `bkt/params.csv`.

Before committing: **Clear all outputs**.